<a href="https://colab.research.google.com/github/Ajaysunil2003/ECE-570-Checkpont-3/blob/main/Checkpoint3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#BLOCK 1

# Adaptive Enhancement for Low-Light Image Restoration
# Original implementation based on concepts from:
# - Xu et al. (2020): Frequency-based decomposition approach
# - Zamir et al. (2022): Efficient transformer for high-resolution images
# - Luo et al. (2024): Content-specific feature extraction

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
import requests
from io import BytesIO
import cv2
import os
import torchvision.models as models
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Install required packages
!pip install lpips

# Import lpips after installation
import lpips

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#BLOCK 2

# ----- Dataset Loading and Processing -----

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Search for the LOLdataset.zip file in Google Drive
import glob

# First, let's look for the file in common locations
possible_locations = [
    '/content/drive/MyDrive/LOLdataset.zip',
    '/content/drive/MyDrive/*/LOLdataset.zip',
    '/content/drive/Starred/LOLdataset.zip',
    '/content/drive/Shareddrives/*/LOLdataset.zip'
]

found_zip = None
for pattern in possible_locations:
    matches = glob.glob(pattern)
    if matches:
        found_zip = matches[0]
        break

# If not found in common locations, do a more extensive search
if not found_zip:
    print("Searching for LOLdataset.zip in your Google Drive (this might take a while)...")
    matches = glob.glob('/content/drive/**/*LOLdataset.zip', recursive=True)
    if matches:
        found_zip = matches[0]

# Report results
if found_zip:
    print(f"Found LOLdataset.zip at: {found_zip}")
    zip_file_path = found_zip
else:
    print("Could not find LOLdataset.zip in your Google Drive")
    # List contents of Starred folder to help locate the file
    print("\nContents of Starred folder:")
    !ls -la /content/drive/Starred/

    # Ask for manual path input
    print("\nPlease check the exact path of your file and update the code if needed.")
    # For now, we'll proceed with synthetic dataset
    zip_file_path = None

# Create local directory for dataset
!mkdir -p data/LOL

# Extract the dataset if found
if zip_file_path:
    print(f"Extracting LOL dataset from {zip_file_path}...")
    !unzip -q -o "{zip_file_path}" -d data/LOL

    # Check the extracted structure
    print("Checking extracted structure:")
    !ls -la data/LOL
else:
    print("Using synthetic dataset since LOLdataset.zip was not found.")

# Try to locate the dataset directories
lol_base_path = 'data/LOL'

# Check for our485 and eval15 directories
if os.path.exists(lol_base_path):
    if not (os.path.exists(os.path.join(lol_base_path, 'our485')) and
            os.path.exists(os.path.join(lol_base_path, 'eval15'))):
        # Look for nested directories
        subdirs = [d for d in os.listdir(lol_base_path) if os.path.isdir(os.path.join(lol_base_path, d))]
        if subdirs:
            print(f"Found subdirectories: {subdirs}")
            # Try first subdirectory
            lol_base_path = os.path.join(lol_base_path, subdirs[0])
            print(f"Checking in {lol_base_path}")
            !ls -la {lol_base_path}

# Define dataset paths
train_low_dir = os.path.join(lol_base_path, 'our485/low/')
train_high_dir = os.path.join(lol_base_path, 'our485/high/')
test_low_dir = os.path.join(lol_base_path, 'eval15/low/')
test_high_dir = os.path.join(lol_base_path, 'eval15/high/')

# Verify all directories exist
all_dirs_exist = True
for dir_path in [train_low_dir, train_high_dir, test_low_dir, test_high_dir]:
    if os.path.exists(dir_path):
        print(f"{dir_path} exists with {len(os.listdir(dir_path))} files")
    else:
        print(f"Warning: {dir_path} does not exist")
        all_dirs_exist = False

# Fall back to synthetic dataset if directories not found
if not all_dirs_exist:
    print("LOL dataset directories not found. Creating synthetic dataset instead...")

    def create_synthetic_dataset():
        import numpy as np
        from PIL import Image
        import os

        # Create directories
        os.makedirs('data/synthetic/our485/low', exist_ok=True)
        os.makedirs('data/synthetic/our485/high', exist_ok=True)
        os.makedirs('data/synthetic/eval15/low', exist_ok=True)
        os.makedirs('data/synthetic/eval15/high', exist_ok=True)

        # Create 20 training pairs and 5 test pairs
        for i in range(20):
            # Create a random "normal" image
            img = np.random.rand(256, 256, 3)
            high_img = Image.fromarray((img * 255).astype('uint8'))
            high_img.save(f'data/synthetic/our485/high/{i:03d}.png')

            # Create a darker, noisier version for low-light
            low_img = Image.fromarray(((img * 0.3 + np.random.rand(256, 256, 3) * 0.05) * 255).astype('uint8'))
            low_img.save(f'data/synthetic/our485/low/{i:03d}.png')

        for i in range(5):
            # Create a random "normal" image
            img = np.random.rand(256, 256, 3)
            high_img = Image.fromarray((img * 255).astype('uint8'))
            high_img.save(f'data/synthetic/eval15/high/{i:03d}.png')

            # Create a darker, noisier version for low-light
            low_img = Image.fromarray(((img * 0.3 + np.random.rand(256, 256, 3) * 0.05) * 255).astype('uint8'))
            low_img.save(f'data/synthetic/eval15/low/{i:03d}.png')

        return 'data/synthetic/our485/low/', 'data/synthetic/our485/high/', 'data/synthetic/eval15/low/', 'data/synthetic/eval15/high/'

    train_low_dir, train_high_dir, test_low_dir, test_high_dir = create_synthetic_dataset()

# Create dataset class
class LOLDataset(torch.utils.data.Dataset):
    def __init__(self, low_dir, high_dir, transform=None):
        if not os.path.exists(low_dir):
            raise FileNotFoundError(f"Low-light directory not found: {low_dir}")
        if not os.path.exists(high_dir):
            raise FileNotFoundError(f"High-light directory not found: {high_dir}")

        # Find all image files
        low_files = []
        for ext in ['.png', '.jpg', '.jpeg']:
            low_files.extend(glob.glob(os.path.join(low_dir, f'*{ext}')))

        high_files = []
        for ext in ['.png', '.jpg', '.jpeg']:
            high_files.extend(glob.glob(os.path.join(high_dir, f'*{ext}')))

        if len(low_files) == 0 or len(high_files) == 0:
            print(f"Warning: No image files found. Low: {len(low_files)}, High: {len(high_files)}")

        # Sort files to ensure consistent pairing
        self.low_files = sorted(low_files)
        self.high_files = sorted(high_files)

        # Make sure we have the same number of files
        min_files = min(len(self.low_files), len(self.high_files))
        self.low_files = self.low_files[:min_files]
        self.high_files = self.high_files[:min_files]

        self.transform = transform
        print(f"Loaded dataset with {len(self.low_files)} image pairs")

    def __len__(self):
        return len(self.low_files)

    def __getitem__(self, idx):
        try:
            low_img = Image.open(self.low_files[idx]).convert('RGB')
            high_img = Image.open(self.high_files[idx]).convert('RGB')

            if self.transform:
                low_img = self.transform(low_img)
                high_img = self.transform(high_img)

            return low_img, high_img
        except Exception as e:
            print(f"Error loading image pair {idx}: {e}")
            # Return a small blank image pair in case of error
            blank = torch.zeros(3, 256, 256)
            return blank, blank

# Set up transforms
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# Try to create datasets
try:
    train_dataset = LOLDataset(train_low_dir, train_high_dir, transform=transform)
    test_dataset = LOLDataset(test_low_dir, test_high_dir, transform=transform)
    print("Successfully loaded dataset")
except FileNotFoundError as e:
    print(f"Error loading dataset: {e}")
    print("Creating synthetic dataset as fallback...")
    train_low_dir, train_high_dir, test_low_dir, test_high_dir = create_synthetic_dataset()
    train_dataset = LOLDataset(train_low_dir, train_high_dir, transform=transform)
    test_dataset = LOLDataset(test_low_dir, test_high_dir, transform=transform)
    print("Successfully loaded synthetic dataset")

# Create dataloaders with fewer workers for Colab compatibility
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
print(f"Created train_loader with {len(train_loader)} batches and test_loader with {len(test_loader)} batches")

In [ ]:
#BLOCK 3

# Display some sample images from the dataset
def show_sample_images(dataset, num_samples=3):
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 5 * num_samples))

    for i in range(num_samples):
        idx = np.random.randint(0, len(dataset))
        low_img, high_img = dataset[idx]

        # Convert tensors to numpy images
        low_np = low_img.permute(1, 2, 0).numpy()
        high_np = high_img.permute(1, 2, 0).numpy()

        axes[i, 0].imshow(low_np)
        axes[i, 0].set_title(f'Low-light Image {i+1}')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(high_np)
        axes[i, 1].set_title(f'Ground Truth {i+1}')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

print("Displaying sample images from LOL dataset:")
show_sample_images(train_dataset)

In [ ]:
#BLOCK 4

# 1. Content Analysis Module (Based on concept from Luo et al., 2024)
# This module analyzes different regions of the image to extract characteristics
class ContentAnalysisModule(nn.Module):
    def __init__(self, in_channels=3, feature_dim=64):
        super(ContentAnalysisModule, self).__init__()
        # Base encoder extracts general features
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True)
        )

        # Specialized analyzers for different image characteristics
        self.brightness_analyzer = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, feature_dim//2, kernel_size=3, padding=1)
        )

        self.noise_analyzer = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, feature_dim//2, kernel_size=3, padding=1)
        )

        self.contrast_analyzer = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, feature_dim//2, kernel_size=3, padding=1)
        )

    def forward(self, x):
        # Extract general features
        features = self.encoder(x)

        # Extract specific characteristics
        brightness_map = self.brightness_analyzer(features)
        noise_map = self.noise_analyzer(features)
        contrast_map = self.contrast_analyzer(features)

        return {
            'features': features,
            'brightness': brightness_map,
            'noise': noise_map,
            'contrast': contrast_map
        }

In [ ]:
#BLOCK 5

# 2. Attention to Context Encoding (ACE) Module (Based on Xu et al., 2020)
# This module decomposes the image into frequency components
class ACEModule(nn.Module):
    def __init__(self, channels):
        super(ACEModule, self).__init__()
        # Different receptive fields to capture different frequencies
        self.dilated_conv1 = nn.Conv2d(channels, channels, kernel_size=1, dilation=1, padding=0)
        self.dilated_conv2 = nn.Conv2d(channels, channels, kernel_size=3, dilation=2, padding=2)

        # Non-local context encoding
        self.context_conv1 = nn.Conv2d(channels, channels//2, kernel_size=1)
        self.context_conv2 = nn.Conv2d(channels, channels//2, kernel_size=1)
        self.context_conv3 = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch_size, c, h, w = x.shape

        # Extract different frequency information (as in Xu et al.)
        feat_d1 = self.dilated_conv1(x)
        feat_d2 = self.dilated_conv2(x)

        # Compute contrast-aware attention map
        contrast_map = torch.sigmoid(feat_d1 - feat_d2)
        inverse_map = 1 - contrast_map

        # Select low-frequency content
        low_freq = inverse_map * x

        # Select high-frequency content
        high_freq = contrast_map * x

        # Non-local context encoding (adapting Xu's approach)
        # Reshape for matrix multiplication
        query = self.context_conv1(low_freq).view(batch_size, -1, h*w).permute(0, 2, 1)  # B x (h*w) x c//2
        key = self.context_conv2(low_freq).view(batch_size, -1, h*w)  # B x c//2 x (h*w)

        # Compute attention weights
        energy = torch.bmm(query, key)  # B x (h*w) x (h*w)
        attention = F.softmax(energy / (c ** 0.5), dim=2)

        # Apply attention weights
        value = x.view(batch_size, c, -1)  # B x c x (h*w)
        out = torch.bmm(value, attention.permute(0, 2, 1))
        out = out.view(batch_size, c, h, w)
        out = self.context_conv3(out)

        # Residual connection
        out = out + low_freq

        return low_freq, high_freq, contrast_map, out

In [ ]:
#BLOCK 6

# 3. Parameter Prediction Network (Original Implementation)
class ParameterPredictionNetwork(nn.Module):
    def __init__(self, feature_dim=64):
        super(ParameterPredictionNetwork, self).__init__()
        # Process combined analysis features
        self.conv_layers = nn.Sequential(
            nn.Conv2d(feature_dim * 3 // 2, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True)
        )

        # Spatial Attention for localized parameter prediction
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, 1, kernel_size=1),
            nn.Sigmoid()
        )

        # Parameter prediction heads
        self.brightness_head = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, 1, kernel_size=1)
        )

        self.contrast_head = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, 1, kernel_size=1)
        )

        self.denoise_head = nn.Sequential(
            nn.Conv2d(feature_dim, feature_dim//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim//2, 1, kernel_size=1)
        )

    def forward(self, brightness, noise, contrast):
        # Concatenate all feature maps
        combined = torch.cat([brightness, noise, contrast], dim=1)
        features = self.conv_layers(combined)

        # Apply spatial attention
        attention_map = self.spatial_attention(features)
        attended_features = features * attention_map

        # Generate enhancement parameters
        brightness_factor = torch.sigmoid(self.brightness_head(attended_features)) * 3.0 + 0.5  # Range: 0.5-3.5
        contrast_factor = torch.sigmoid(self.contrast_head(attended_features)) * 2.0 + 0.5  # Range: 0.5-2.5
        denoise_strength = torch.sigmoid(self.denoise_head(attended_features))

        return {
            'brightness': brightness_factor,
            'contrast': contrast_factor,
            'denoise': denoise_strength,
            'attention': attention_map
        }

In [ ]:
#BLOCK 7

# 4. Multi-Dconv Head Transposed Attention (MDTA) (Based on Zamir et al., 2022)
class MDTABlock(nn.Module):
    def __init__(self, channels=64, num_heads=4):
        super(MDTABlock, self).__init__()
        self.num_heads = num_heads
        self.channels_per_head = channels // num_heads
        self.scale = self.channels_per_head ** -0.5

        # Layer normalization
        self.norm = nn.LayerNorm(channels)

        # Point-wise convolutions for query, key, value
        self.qkv_proj = nn.Conv2d(channels, channels*3, kernel_size=1, bias=False)

        # Depth-wise convolutions for local context
        self.dw_conv_q = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)
        self.dw_conv_k = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)
        self.dw_conv_v = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)

        # Output projection
        self.proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        batch_size, channels, height, width = x.shape
        shortcut = x

        # Layer normalization
        x_flat = x.permute(0, 2, 3, 1).contiguous()  # B, H, W, C
        x_norm = self.norm(x_flat).permute(0, 3, 1, 2).contiguous()  # B, C, H, W

        # Generate Q, K, V with point-wise convolutions
        qkv = self.qkv_proj(x_norm)
        q, k, v = qkv.chunk(3, dim=1)

        # Apply depth-wise convolutions for local context
        q = self.dw_conv_q(q)
        k = self.dw_conv_k(k)
        v = self.dw_conv_v(v)

        # Reshape for multi-head attention
        q = q.reshape(batch_size, self.num_heads, self.channels_per_head, height * width)
        k = k.reshape(batch_size, self.num_heads, self.channels_per_head, height * width)
        v = v.reshape(batch_size, self.num_heads, self.channels_per_head, height * width)

        # Transposed attention (channel-wise instead of spatial)
        # Different from standard attention, this operates across channels like in Zamir et al.
        q = q.permute(0, 1, 3, 2)  # B, heads, H*W, channels_per_head

        # Calculate attention scores and apply to values
        attn = torch.matmul(q, k) * self.scale  # B, heads, H*W, H*W
        attn = F.softmax(attn, dim=-1)

        # Apply attention weights
        out = torch.matmul(attn, v.permute(0, 1, 3, 2))  # B, heads, H*W, channels_per_head
        out = out.permute(0, 1, 3, 2).reshape(batch_size, channels, height, width)

        # Output projection and residual connection
        out = self.proj(out)
        return out + shortcut

# 5. Gated-Dconv Feed-Forward Network (GDFN) (Based on Zamir et al., 2022)
class GDFNBlock(nn.Module):
    def __init__(self, channels=64, expansion_factor=2.0):
        super(GDFNBlock, self).__init__()

        hidden_channels = int(channels * expansion_factor)

        # Layer normalization
        self.norm = nn.LayerNorm(channels)

        # Feed-forward with gating
        self.conv1 = nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=False)
        self.conv2 = nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=False)

        # Depth-wise convolution for spatial context
        self.dw_conv1 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3,
                                 padding=1, groups=hidden_channels, bias=False)
        self.dw_conv2 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3,
                                 padding=1, groups=hidden_channels, bias=False)

        # Output projection
        self.proj = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

        # GELU activation
        self.gelu = nn.GELU()

    def forward(self, x):
        shortcut = x

        # Layer normalization
        x_flat = x.permute(0, 2, 3, 1).contiguous()  # B, H, W, C
        x_norm = self.norm(x_flat).permute(0, 3, 1, 2).contiguous()  # B, C, H, W

        # Split into two paths for gating
        x1 = self.conv1(x_norm)
        x1 = self.dw_conv1(x1)
        x1 = self.gelu(x1)

        x2 = self.conv2(x_norm)
        x2 = self.dw_conv2(x2)

        # Gating operation
        x = x1 * x2

        # Output projection and residual connection
        x = self.proj(x)
        return x + shortcut

In [ ]:
#BLOCK 8

# 6. Dynamic Enhancement Layer (Original Implementation)
class DynamicEnhancementLayer(nn.Module):
    def __init__(self):
        super(DynamicEnhancementLayer, self).__init__()

    def forward(self, image, params):
        # Apply region-specific enhancements based on predicted parameters
        enhanced = image.clone()

        # Apply brightness adjustment
        enhanced = enhanced * params['brightness']

        # Apply contrast adjustment
        mean = torch.mean(enhanced, dim=[2, 3], keepdim=True)
        enhanced = (enhanced - mean) * params['contrast'] + mean

        # Apply denoising (soft thresholding approach)
        noise_mask = 1.0 - params['denoise']
        enhanced = enhanced * noise_mask + torch.tanh(enhanced) * params['denoise']

        return enhanced.clamp(0, 1)

# 7. Cross Domain Transformation (CDT) Module (Based on Xu et al., 2020)
class CDTModule(nn.Module):
    def __init__(self, channels):
        super(CDTModule, self).__init__()

        # Channel attention mechanism
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels*2, channels//2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels//2, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, encoder_features, decoder_features, contrast_map):
        # Apply contrast map to weight encoder features
        weighted_encoder = encoder_features * (1.0 - contrast_map)

        # Concatenate features
        combined = torch.cat([weighted_encoder, decoder_features], dim=1)

        # Generate channel attention weights
        weights = self.channel_gate(combined)

        # Apply weights to decoder features
        enhanced = decoder_features * weights

        return enhanced

In [ ]:
#BLOCK 9

# 8. Complete Adaptive Enhancement Network (AENet)
class AdaptiveEnhancementNetwork(nn.Module):
    def __init__(self, in_channels=3, feature_dim=64):
        super(AdaptiveEnhancementNetwork, self).__init__()

        # Initial feature extraction
        self.init_conv = nn.Conv2d(in_channels, feature_dim, kernel_size=3, padding=1)

        # Content analysis stream
        self.content_analyzer = ContentAnalysisModule(in_channels, feature_dim)

        # Encoder blocks with MDTA and GDFN
        self.encoder_blocks = nn.ModuleList([
            nn.Sequential(
                MDTABlock(feature_dim, num_heads=2),
                GDFNBlock(feature_dim)
            ),
            nn.Sequential(
                MDTABlock(feature_dim, num_heads=2),
                GDFNBlock(feature_dim)
            )
        ])

        # ACE module for frequency decomposition
        self.ace_module = ACEModule(feature_dim)

        # Parameter prediction
        self.param_predictor = ParameterPredictionNetwork(feature_dim)

        # Decoder blocks
        self.decoder_blocks = nn.ModuleList([
            nn.Sequential(
                MDTABlock(feature_dim, num_heads=2),
                GDFNBlock(feature_dim)
            ),
            nn.Sequential(
                MDTABlock(feature_dim, num_heads=2),
                GDFNBlock(feature_dim)
            )
        ])

        # CDT modules for skip connections
        self.cdt_modules = nn.ModuleList([
            CDTModule(feature_dim),
            CDTModule(feature_dim)
        ])

        # Dynamic enhancement
        self.dynamic_enhancement = DynamicEnhancementLayer()

        # Final output layer
        self.output_conv = nn.Conv2d(feature_dim, in_channels, kernel_size=3, padding=1)

    def forward(self, x):
        # Initial feature extraction
        init_features = self.init_conv(x)

        # Content analysis
        analysis_results = self.content_analyzer(x)

        # Encoder path
        encoder_features = [init_features]
        current = init_features

        for block in self.encoder_blocks:
            current = block(current)
            encoder_features.append(current)

        # Frequency decomposition
        low_freq, high_freq, contrast_map, context_enhanced = self.ace_module(current)

        # Parameter prediction
        params = self.param_predictor(
            analysis_results['brightness'],
            analysis_results['noise'],
            analysis_results['contrast']
        )

        # Two-stage enhancement (adapting Xu's approach)
        # Stage 1: Low-frequency enhancement
        enhanced_low_freq = self.dynamic_enhancement(low_freq, params)

        # Decoder path with skip connections
        current = enhanced_low_freq

        for i, block in enumerate(self.decoder_blocks):
            skip_connection = encoder_features[-(i+1)]
            current = self.cdt_modules[i](skip_connection, current, contrast_map)
            current = block(current)

        # Stage 2: High-frequency detail refinement
        high_freq_refined = high_freq * (1.0 - params['denoise'])
        current = current + high_freq_refined

        # Final output
        output = self.output_conv(current)
        return torch.sigmoid(output), params

In [ ]:
#BLOCK 10

# ----- Evaluation Metrics -----

def calculate_psnr(img1, img2):
    """Calculate PSNR between two images"""
    # Convert to numpy if tensors
    if torch.is_tensor(img1):
        img1 = img1.detach().cpu().numpy().transpose(1, 2, 0)
    if torch.is_tensor(img2):
        img2 = img2.detach().cpu().numpy().transpose(1, 2, 0)

    return psnr(img1, img2, data_range=1.0)

def calculate_ssim(img1, img2):
    """Calculate SSIM between two images"""
    # Convert to numpy if tensors
    if torch.is_tensor(img1):
        img1 = img1.detach().cpu().numpy().transpose(1, 2, 0)
    if torch.is_tensor(img2):
        img2 = img2.detach().cpu().numpy().transpose(1, 2, 0)

    # For multi-channel images
    if img1.shape[2] > 1:
        multichannel = True
    else:
        multichannel = False

    return ssim(img1, img2, multichannel=multichannel, data_range=1.0)

# Initialize LPIPS metric
loss_fn_alex = lpips.LPIPS(net='alex')

def calculate_lpips(img1, img2):
    """Calculate LPIPS between two images"""
    # Make sure inputs are tensors
    if not torch.is_tensor(img1):
        img1 = transforms.ToTensor()(img1).unsqueeze(0)
    else:
        img1 = img1.unsqueeze(0) if img1.dim() == 3 else img1

    if not torch.is_tensor(img2):
        img2 = transforms.ToTensor()(img2).unsqueeze(0)
    else:
        img2 = img2.unsqueeze(0) if img2.dim() == 3 else img2

    return loss_fn_alex(img1, img2).item()

In [ ]:
#BLOCK 11

# ----- Model Training Function -----

def train_model(model, train_loader, val_loader, num_epochs=5):
    # Loss functions
    l1_loss = nn.L1Loss()
    mse_loss = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_psnr': [],
        'val_ssim': []
    }

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0

        for batch_idx, (low_img, high_img) in enumerate(train_loader):
            low_img, high_img = low_img.to(device), high_img.to(device)

            # Forward pass
            enhanced_img, params = model(low_img)

            # Calculate losses
            reconstruction_loss = l1_loss(enhanced_img, high_img)

            # Parameter smoothness loss (to prevent artifacts)
            smoothness_loss = 0.01 * (
                torch.mean(torch.abs(params['brightness'][:, :, 1:, :] - params['brightness'][:, :, :-1, :])) +
                torch.mean(torch.abs(params['brightness'][:, :, :, 1:] - params['brightness'][:, :, :, :-1])) +
                torch.mean(torch.abs(params['contrast'][:, :, 1:, :] - params['contrast'][:, :, :-1, :])) +
                torch.mean(torch.abs(params['contrast'][:, :, :, 1:] - params['contrast'][:, :, :, :-1])) +
                torch.mean(torch.abs(params['denoise'][:, :, 1:, :] - params['denoise'][:, :, :-1, :])) +
                torch.mean(torch.abs(params['denoise'][:, :, :, 1:] - params['denoise'][:, :, :, :-1]))
            )

            # Combined loss
            loss = reconstruction_loss + smoothness_loss

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            # Print progress
            if (batch_idx + 1) % 20 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        # Average epoch loss
        epoch_loss /= len(train_loader)
        history['train_loss'].append(epoch_loss)

        # Validation
        model.eval()
        val_loss = 0
        val_psnr = 0
        val_ssim = 0

        with torch.no_grad():
            for low_img, high_img in val_loader:
                low_img, high_img = low_img.to(device), high_img.to(device)

                # Forward pass
                enhanced_img, _ = model(low_img)

                # Calculate loss
                loss = l1_loss(enhanced_img, high_img)
                val_loss += loss.item()

                # Calculate metrics
                psnr_val = calculate_psnr(enhanced_img[0].cpu(), high_img[0].cpu())
                ssim_val = calculate_ssim(enhanced_img[0].cpu(), high_img[0].cpu())

                val_psnr += psnr_val
                val_ssim += ssim_val

        # Average validation metrics
        val_loss /= len(val_loader)
        val_psnr /= len(val_loader)
        val_ssim /= len(val_loader)

        history['val_loss'].append(val_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)

        # Update learning rate
        scheduler.step(val_loss)

        # Print epoch summary
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}')

        # Save model checkpoint
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'aenet_checkpoint_epoch_{epoch+1}.pth')

    # Save final model
    torch.save(model.state_dict(), 'aenet_final.pth')

    return model, history

In [ ]:
#BLOCK 12

# ----- Visualization Functions -----

def visualize_enhancement_parameters(params, original_img):
    """Visualize the predicted enhancement parameters"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Display original image
    axes[0, 0].imshow(original_img.permute(1, 2, 0).cpu().numpy())
    axes[0, 0].set_title('Original Low-Light Image')
    axes[0, 0].axis('off')

    # Display brightness parameter map
    brightness_map = params['brightness'][0, 0].cpu().numpy()
    im = axes[0, 1].imshow(brightness_map, cmap='inferno')
    axes[0, 1].set_title('Brightness Enhancement Map')
    axes[0, 1].axis('off')
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

    # Display contrast parameter map
    contrast_map = params['contrast'][0, 0].cpu().numpy()
    im = axes[1, 0].imshow(contrast_map, cmap='viridis')
    axes[1, 0].set_title('Contrast Enhancement Map')
    axes[1, 0].axis('off')
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

    # Display denoising parameter map
    denoise_map = params['denoise'][0, 0].cpu().numpy()
    im = axes[1, 1].imshow(denoise_map, cmap='plasma')
    axes[1, 1].set_title('Denoising Strength Map')
    axes[1, 1].axis('off')
    plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

def display_results(model, dataset, num_samples=5, indices=None):
    """Display before/after comparisons with enhancement parameters"""
    model.eval()

    if indices is None:
        # Random selection
        indices = np.random.choice(len(dataset), num_samples, replace=False)

    for idx in indices:
        low_img, high_img = dataset[idx]
        low_img = low_img.unsqueeze(0).to(device)
        high_img = high_img.unsqueeze(0).to(device)

        # Generate enhanced image
        with torch.no_grad():
            enhanced_img, params = model(low_img)

        # Calculate metrics
        psnr_val = calculate_psnr(enhanced_img[0].cpu(), high_img[0].cpu())
        ssim_val = calculate_ssim(enhanced_img[0].cpu(), high_img[0].cpu())
        lpips_val = calculate_lpips(enhanced_img[0].cpu(), high_img[0].cpu())

        # Display images
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(low_img[0].permute(1, 2, 0).cpu().numpy())
        plt.title('Low-Light Input')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(enhanced_img[0].permute(1, 2, 0).cpu().numpy())
        plt.title(f'Enhanced Image\nPSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}, LPIPS: {lpips_val:.4f}')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(high_img[0].permute(1, 2, 0).cpu().numpy())
        plt.title('Ground Truth')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

        # Visualize enhancement parameters
        visualize_enhancement_parameters(params, low_img[0])

In [ ]:
#BLOCK 13

def analyze_mixed_lighting(model, img_path):
    """Analyze performance on a mixed lighting image"""
    # Load image
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Process with model
    with torch.no_grad():
        enhanced_img, params = model(img_tensor)

    # Display results
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_tensor[0].permute(1, 2, 0).cpu().numpy())
    plt.title('Mixed Lighting Input')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(enhanced_img[0].permute(1, 2, 0).cpu().numpy())
    plt.title('Enhanced Result')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    # Visualize parameter maps
    visualize_enhancement_parameters(params, img_tensor[0])

    # Analyze different regions
    height, width = params['brightness'][0, 0].shape

    # Define regions (top-left, top-right, bottom-left, bottom-right)
    regions = [
        (0, 0, height//2, width//2),
        (0, width//2, height//2, width),
        (height//2, 0, height, width//2),
        (height//2, width//2, height, width)
    ]

    region_names = ['Top-left', 'Top-right', 'Bottom-left', 'Bottom-right']

    # Compute average parameter values for each region
    print("Region-specific Enhancement Analysis:")
    for i, (y1, x1, y2, x2) in enumerate(regions):
        avg_brightness = params['brightness'][0, 0, y1:y2, x1:x2].mean().item()
        avg_contrast = params['contrast'][0, 0, y1:y2, x1:x2].mean().item()
        avg_denoise = params['denoise'][0, 0, y1:y2, x1:x2].mean().item()

        print(f"{region_names[i]} Region:")
        print(f"  Average Brightness Factor: {avg_brightness:.4f}")
        print(f"  Average Contrast Factor: {avg_contrast:.4f}")
        print(f"  Average Denoising Strength: {avg_denoise:.4f}")
        print()

In [ ]:
#BLOCK 14


# ----- Compare with Fixed-Parameter Approaches -----

class FixedParameterModel(nn.Module):
    """A simplified model using fixed enhancement parameters"""
    def __init__(self):
        super(FixedParameterModel, self).__init__()

        # Fixed enhancement network
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.features(x)

def compare_with_fixed_parameter(adaptive_model, test_dataset, indices=None):
    """Compare adaptive approach with fixed-parameter approach"""
    # Create and train a fixed parameter model
    fixed_model = FixedParameterModel().to(device)

    # For demonstration, we'll just load a pre-trained fixed model
    # In practice, you would train this model on the same dataset
    # fixed_model.load_state_dict(torch.load('fixed_model.pth'))

    # For this demo, we'll just use a simple model without training

    if indices is None:
        # Select specific test cases that highlight differences
        indices = [5, 10, 15]  # Choose appropriate indices

    # Evaluate both models
    adaptive_model.eval()
    fixed_model.eval()

    metrics = {
        'adaptive': {'psnr': [], 'ssim': [], 'lpips': []},
        'fixed': {'psnr': [], 'ssim': [], 'lpips': []}
    }

    for idx in indices:
        low_img, high_img = test_dataset[idx]
        low_img = low_img.unsqueeze(0).to(device)
        high_img = high_img.unsqueeze(0).to(device)

        # Generate enhanced images
        with torch.no_grad():
            adaptive_enhanced, params = adaptive_model(low_img)
            fixed_enhanced = fixed_model(low_img)

        # Calculate metrics
        for model_type, enhanced in [('adaptive', adaptive_enhanced), ('fixed', fixed_enhanced)]:
            psnr_val = calculate_psnr(enhanced[0].cpu(), high_img[0].cpu())
            ssim_val = calculate_ssim(enhanced[0].cpu(), high_img[0].cpu())
            lpips_val = calculate_lpips(enhanced[0].cpu(), high_img[0].cpu())

            metrics[model_type]['psnr'].append(psnr_val)
            metrics[model_type]['ssim'].append(ssim_val)
            metrics[model_type]['lpips'].append(lpips_val)

        # Display comparison
        plt.figure(figsize=(20, 5))

        plt.subplot(1, 4, 1)
        plt.imshow(low_img[0].permute(1, 2, 0).cpu().numpy())
        plt.title('Low-Light Input')
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.imshow(adaptive_enhanced[0].permute(1, 2, 0).cpu().numpy())
        plt.title(f'Adaptive Enhancement\nPSNR: {metrics["adaptive"]["psnr"][-1]:.2f}, SSIM: {metrics["adaptive"]["ssim"][-1]:.4f}')
        plt.axis('off')

        plt.subplot(1, 4, 3)
        plt.imshow(fixed_enhanced[0].permute(1, 2, 0).cpu().numpy())
        plt.title(f'Fixed Parameter Enhancement\nPSNR: {metrics["fixed"]["psnr"][-1]:.2f}, SSIM: {metrics["fixed"]["ssim"][-1]:.4f}')
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.imshow(high_img[0].permute(1, 2, 0).cpu().numpy())
        plt.title('Ground Truth')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

        # Show enhancement parameters
        visualize_enhancement_parameters(params, low_img[0])

    # Calculate average metrics
    print("Average Metrics Comparison:")
    for metric in ['psnr', 'ssim', 'lpips']:
        adaptive_avg = np.mean(metrics['adaptive'][metric])
        fixed_avg = np.mean(metrics['fixed'][metric])

        better = "Better" if ((metric == 'lpips' and adaptive_avg < fixed_avg) or
                             (metric != 'lpips' and adaptive_avg > fixed_avg)) else "Worse"

        print(f"{metric.upper()}: Adaptive = {adaptive_avg:.4f}, Fixed = {fixed_avg:.4f} ({better})")

In [ ]:
#BLOCK 15

# ----- Main Execution -----

# Free up memory
import torch
import gc

# Empty CUDA cache
torch.cuda.empty_cache()

# Run garbage collection
gc.collect()

# Print available GPU memory
print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Reduce model size significantly
model = AdaptiveEnhancementNetwork(feature_dim=8).to(device)  # Very small model for demo
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

# Process just a single 128x128 image for demonstration
with torch.no_grad():  # Important to prevent memory allocation for gradients
    if len(test_dataset) > 0:
        low_img, high_img = test_dataset[0]

        # Reduce resolution for processing
        low_small = transforms.Resize((128, 128))(low_img).unsqueeze(0).to(device)

        # Forward pass with smaller image
        enhanced_img, params = model(low_small)

        # Display results
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(low_img.permute(1, 2, 0).cpu().numpy())
        plt.title('Low-Light Input')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(enhanced_img[0].permute(1, 2, 0).detach().cpu().numpy())
        plt.title('Enhanced (Demo Only)')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(high_img.permute(1, 2, 0).cpu().numpy())
        plt.title('Ground Truth')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

        # Show just one parameter map
        plt.figure(figsize=(5, 5))
        plt.imshow(params['brightness'][0, 0].detach().cpu().numpy(), cmap='inferno')
        plt.colorbar()
        plt.title('Brightness Enhancement Map')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

        print("Model demonstration complete - Showing minimal example due to memory constraints")
        print("For full implementation, additional memory optimization would be required")

In [ ]:
#BLOCK 16

# ----- Additional Visualization Functions -----

def visualize_training_history(history):
    """Visualize training and validation metrics over epochs"""
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(15, 5))

    # Loss plot
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # PSNR plot
    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['val_psnr'], 'g-')
    plt.title('Validation PSNR')
    plt.xlabel('Epochs')
    plt.ylabel('PSNR (dB)')

    # SSIM plot
    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['val_ssim'], 'm-')
    plt.title('Validation SSIM')
    plt.xlabel('Epochs')
    plt.ylabel('SSIM')

    plt.tight_layout()
    plt.show()

# Function to collect example results for your paper/presentation
def collect_paper_examples(model, dataset, indices=None, save_path='paper_examples/'):
    """Save before/after examples with parameter maps for paper"""
    os.makedirs(save_path, exist_ok=True)
    model.eval()

    if indices is None:
        # Select good demonstration examples
        indices = [0, 1, 2]  # Adjust based on your dataset

    results = []
    for i, idx in enumerate(indices):
        low_img, high_img = dataset[idx]
        low_img = low_img.unsqueeze(0).to(device)
        high_img = high_img.unsqueeze(0).to(device)

        # Generate enhanced image
        with torch.no_grad():
            enhanced_img, params = model(low_img)

        # Calculate metrics
        psnr_val = calculate_psnr(enhanced_img[0].cpu(), high_img[0].cpu())
        ssim_val = calculate_ssim(enhanced_img[0].cpu(), high_img[0].cpu())
        lpips_val = calculate_lpips(enhanced_img[0].cpu(), high_img[0].cpu())

        # Save results
        results.append({
            'low_img': low_img[0].cpu(),
            'enhanced_img': enhanced_img[0].cpu(),
            'high_img': high_img[0].cpu(),
            'params': {k: v[0].cpu() for k, v in params.items()},
            'metrics': {
                'psnr': psnr_val,
                'ssim': ssim_val,
                'lpips': lpips_val
            }
        })

        # Save individual images
        plt.imsave(f'{save_path}example_{i}_input.png',
                  low_img[0].permute(1, 2, 0).cpu().numpy())
        plt.imsave(f'{save_path}example_{i}_enhanced.png',
                  enhanced_img[0].permute(1, 2, 0).cpu().numpy())
        plt.imsave(f'{save_path}example_{i}_ground_truth.png',
                  high_img[0].permute(1, 2, 0).cpu().numpy())

        # Save parameter maps for visualization
        for param_name in ['brightness', 'contrast', 'denoise']:
            plt.figure(figsize=(5, 5))
            plt.imshow(params[param_name][0, 0].cpu().numpy(),
                      cmap='viridis' if param_name=='contrast' else
                      ('inferno' if param_name=='brightness' else 'plasma'))
            plt.colorbar()
            plt.title(f'{param_name.capitalize()} Map')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(f'{save_path}example_{i}_{param_name}_map.png', bbox_inches='tight')
            plt.close()

    print(f"Saved {len(indices)} examples to {save_path}")
    return results

# Uncomment to collect examples for your paper
    paper_examples = collect_paper_examples(model, test_dataset)